# 🐉 App Imagen → 3D (Hunyuan3D) — con interfaz web

Corrés 2 celdas una sola vez y obtenés un **link público** (`xxxxx.gradio.live`) donde subís imágenes y bajás modelos 3D desde el navegador, sin tocar código.

1. GPU: `Entorno de ejecución` → `Cambiar tipo` → **T4 GPU**.
2. Celda 1 (instalar) → al terminar, **Reiniciar sesión**.
3. Celda 2 (lanzar app) → te da el link `gradio.live`.

In [ ]:
# Celda 1 — Instalar. Al terminar: Reiniciar sesión.
!nvidia-smi -L
import os
os.chdir('/content')
if not os.path.isdir('/content/Hunyuan3D-2'):
    !git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git
os.chdir('/content/Hunyuan3D-2')
!pip install -q ninja gradio
!pip install -q diffusers transformers accelerate trimesh omegaconf einops opencv-python-headless huggingface_hub
!pip install -q -e . 2>&1 | tail -2
!pip install -q -U 'numpy>=2.1'
print('Instalado. AHORA: Entorno de ejecucion -> Reiniciar sesion, y corre la Celda 2.')

In [ ]:
# Celda 2 — Lanzar la app web (te da el link publico)
import os, torch, gradio as gr
os.chdir('/content/Hunyuan3D-2')
from PIL import Image
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
print('Cargando el modelo (la primera vez baja ~10 GB)...')
pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2')
print('Modelo listo.')

def generar(img, resolucion, pasos):
    if img is None:
        return None, None
    img = img.convert('RGBA')
    mesh = pipe(image=img, num_inference_steps=int(pasos), octree_resolution=int(resolucion), generator=torch.manual_seed(0))[0]
    out = '/content/salida.glb'
    mesh.export(out)
    return out, out

demo = gr.Interface(
    fn=generar,
    inputs=[
        gr.Image(type='pil', label='Imagen del personaje (PNG con fondo transparente = mejor)'),
        gr.Slider(128, 384, value=256, step=32, label='Resolucion (mas = mas detalle)'),
        gr.Slider(20, 50, value=30, step=5, label='Pasos de calidad'),
    ],
    outputs=[gr.Model3D(label='Modelo 3D'), gr.File(label='Descargar .glb')],
    title='Imagen a 3D con Hunyuan3D',
    description='Subi una imagen de personaje y obtene el modelo 3D. Corre en la GPU gratis de Colab.',
    allow_flagging='never',
)
demo.launch(share=True)